# 🧠 Chapter 03-03a: Tools & Output Parsers — Giving LLMs Superpowers

---

## The Story: When LLMs Need to DO Things, Not Just Talk 🔧

So far, our LLMs can only generate text. That's like having a brilliant advisor who can talk but can't use a phone, check the weather, or look anything up.

**Tools** let LLMs take ACTION — search the web, query databases, call APIs, run code. And **Output Parsers** ensure the LLM's responses are structured data your code can actually use.

Together, they transform LLMs from chatbots into **intelligent systems**.

---

## 🎯 What You'll Learn

| Topic | What It Does |
|-------|-------------|
| **Output Parsers** | Convert LLM text into structured data (JSON, dicts, Pydantic) |
| **Tool Definition** | Define functions that LLMs can decide to call |
| **Tool Binding** | Attach tools to a model with `.bind_tools()` |
| **Function Calling** | Let the LLM decide WHEN and HOW to use tools |
| **Custom Tools** | Create your own tools for any use case |

---

## 📤 Output Parsers: The Structured Data Layer

```
╔═══════════════════════════════════════════════════════════════════════╗
║              OUTPUT PARSERS IN LANGCHAIN                              ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  LLMs return TEXT. Your code needs STRUCTURED DATA.                  ║
║  Output parsers bridge this gap.                                     ║
║                                                                       ║
║  StrOutputParser()                                                    ║
║    AIMessage → plain string                                          ║
║    Simplest parser. Extracts .content                                ║
║                                                                       ║
║  JsonOutputParser()                                                   ║
║    AIMessage → Python dict                                           ║
║    Parses JSON from the LLM's text response                         ║
║    Can add format_instructions to the prompt                        ║
║                                                                       ║
║  PydanticOutputParser(pydantic_object=MyModel)                       ║
║    AIMessage → Pydantic model instance                               ║
║    Validates structure AND types                                     ║
║    Auto-generates format instructions for the prompt                ║
║                                                                       ║
║  with_structured_output(MyModel) ← RECOMMENDED!                     ║
║    The modern approach. Built into the model itself.                 ║
║    Uses function calling under the hood.                             ║
║    We covered this in Chapter 2!                                     ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### When to Use Which Parser:

| Parser | Returns | Best For |
|--------|---------|----------|
| `StrOutputParser` | `str` | Simple text output |
| `JsonOutputParser` | `dict` | Quick JSON extraction |
| `PydanticOutputParser` | Pydantic obj | Type-safe structured output |
| `with_structured_output` | Pydantic obj | Production apps (recommended) |

---

## 🔧 Tools: Giving LLMs Abilities

```
╔═══════════════════════════════════════════════════════════════════════╗
║              WHAT ARE TOOLS?                                          ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  A Tool is a FUNCTION that an LLM can decide to call.                ║
║                                                                       ║
║  Components of a Tool:                                                ║
║    • name:        "search_web"                                       ║
║    • description: "Search the internet for current information"      ║
║    • parameters:  {query: str, max_results: int}                    ║
║    • function:    The actual Python code to execute                   ║
║                                                                       ║
║  How it works:                                                        ║
║    1. You define tools and bind them to the model                    ║
║    2. User asks a question                                            ║
║    3. LLM decides IF it needs a tool and WHICH one                   ║
║    4. LLM returns tool name + arguments (NOT the result!)            ║
║    5. YOUR CODE calls the actual function                             ║
║    6. Result feeds back to LLM for final answer                      ║
║                                                                       ║
║  💡 The LLM doesn't execute tools — it decides to use them.         ║
║     Your code does the actual execution. This is a safety feature!   ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📞 Function Calling: The Mechanism

```
╔═══════════════════════════════════════════════════════════════════════╗
║              FUNCTION CALLING FLOW                                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  User: "What's the weather in Tokyo?"                                ║
║                                                                       ║
║  Step 1: LLM sees available tools:                                    ║
║    [get_weather(city: str), search_web(query: str)]                  ║
║                                                                       ║
║  Step 2: LLM DECIDES to call get_weather:                            ║
║    → Returns: {name: "get_weather", args: {city: "Tokyo"}}          ║
║    → This is a ToolCall, NOT text!                                   ║
║                                                                       ║
║  Step 3: YOUR CODE executes get_weather("Tokyo")                     ║
║    → Returns: "22°C, partly cloudy"                                  ║
║                                                                       ║
║  Step 4: Send result back as ToolMessage                             ║
║                                                                       ║
║  Step 5: LLM generates final answer:                                 ║
║    "The weather in Tokyo is 22°C and partly cloudy."                ║
║                                                                       ║
║  💡 In Agents (Ch. 6), this loop runs automatically!                ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Creating Tools in LangChain:

```python
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the internet for current information."""
    # Your search implementation here
    return f"Results for: {query}"

# Bind to model
llm_with_tools = llm.bind_tools([search_web])

# The model can now DECIDE to call search_web
result = llm_with_tools.invoke("What happened in tech today?")
# result.tool_calls → [{name: 'search_web', args: {query: '...'}}]
```

---

## 🛠️ Custom Tool Creation

```
╔═══════════════════════════════════════════════════════════════════════╗
║              CREATING CUSTOM TOOLS                                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Method 1: @tool decorator (simplest)                                 ║
║    @tool                                                              ║
║    def my_tool(arg: str) -> str:                                     ║
║        """Description for the LLM."""  ← CRITICAL!                  ║
║        return result                                                  ║
║                                                                       ║
║  Method 2: StructuredTool (more control)                              ║
║    StructuredTool.from_function(                                      ║
║        func=my_function,                                              ║
║        name="my_tool",                                               ║
║        description="What this does",                                 ║
║        args_schema=MyArgsModel                                       ║
║    )                                                                  ║
║                                                                       ║
║  ⚠️ The docstring/description is what the LLM reads to decide      ║
║     whether to use the tool. Make it clear and specific!            ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Common Tool Patterns:

| Tool | What It Does | Use Case |
|------|-------------|----------|
| Web Search | Searches the internet | Current events, real-time data |
| Calculator | Performs math operations | Exact calculations |
| Database Query | Runs SQL queries | Business data retrieval |
| API Call | Hits external APIs | Weather, stocks, etc. |
| Code Executor | Runs Python code | Dynamic computation |
| File Reader | Reads local files | Document processing |

---

## 🔗 Tools + Output Parsers Together

```
  User Question
       │
       ▼
  LLM (with tools bound)
       │
       ├── No tool needed → Direct text response
       │                     │
       │                     ▼
       │              Output Parser → structured data
       │
       └── Tool needed → ToolCall (name + args)
                          │
                    Execute tool function
                          │
                    ToolMessage (result)
                          │
                    LLM final response
                          │
                    Output Parser → structured data
```

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║           INTERVIEW QUESTIONS FOR THIS MODULE                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: What are tools in LangChain? How do they work?                  ║
║  A: Tools are functions that an LLM can decide to call. You define  ║
║     the tool (name, description, parameters), bind it to a model    ║
║     with .bind_tools(). The LLM returns a ToolCall with the name   ║
║     and args — YOUR code executes it. The LLM never runs tools     ║
║     directly (safety!).                                              ║
║                                                                       ║
║  Q: What is function calling?                                        ║
║  A: A model capability where the LLM outputs structured function   ║
║     calls instead of text. The model sees available functions and   ║
║     their schemas, then decides when to call them. Supported by    ║
║     OpenAI, Anthropic, Google. LangChain wraps this as tools.      ║
║                                                                       ║
║  Q: Output parser vs with_structured_output — which to use?        ║
║  A: with_structured_output (Pydantic) is preferred for production. ║
║     It uses function calling under the hood = more reliable.        ║
║     Output parsers work with prompt-based JSON extraction.          ║
║     Use parsers when model doesn't support function calling.        ║
║                                                                       ║
║  Q: Why is the tool description so important?                        ║
║  A: The LLM reads the description to decide IF and WHEN to use     ║
║     the tool. A bad description = model won't use it correctly.    ║
║     Write it from the LLM's perspective: clear, specific,          ║
║     including when to use and when NOT to use.                      ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Quick Quiz

---

### Question 1:
When a model calls a tool, what does it actually return?

- A) The tool's execution result
- B) The tool name and arguments (ToolCall)
- C) A Python function object
- D) An error message

<details><summary>Answer</summary>

**B) The tool name and arguments (ToolCall)**

The LLM never executes tools. It returns a ToolCall with the function name and arguments. Your application code is responsible for executing the actual function and passing the result back as a ToolMessage.

</details>

---

### Question 2:
Which is the recommended way to get structured output in production?

- A) Regex parsing of LLM text
- B) `StrOutputParser()`
- C) `llm.with_structured_output(PydanticModel)`
- D) Manual `json.loads()` on the response

<details><summary>Answer</summary>

**C) `llm.with_structured_output(PydanticModel)`**

This uses function calling under the hood, giving guaranteed schema compliance. It returns a validated Pydantic object with proper types. Much more reliable than prompt-based JSON extraction.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Output parsers: Str → JSON → Pydantic (increasing reliability)│
├─────────────────────────────────────────────────────────────────────┤
│  📌 with_structured_output = best for production (uses fn calling)│
├─────────────────────────────────────────────────────────────────────┤
│  📌 Tools = functions the LLM can decide to call                  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 @tool decorator + good docstring = custom tool done right     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LLM returns ToolCall, YOUR code executes. Safety first!       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Tool description = how the LLM decides to use it or not       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On with Tools!

We'll build custom tools, bind them to LLMs, handle tool calls, and create an LLM that searches the web and answers!

---

### 🎉 Tools + Output Parsers = Production AI!
LLMs can now DO things, not just talk. This is where AI gets real! 🚀